"""
QC report registrace T2→T1 (novorozenci) – PDF report + CSV metrik

Co skript dělá:
- načte vybrane_snimky.txt (pacient, cesta_T1, cesta_T2)
- pro každého pacienta načte T1 po N4 (z derivatives/coregistered) a T2_reg (registrovanou)
- vytvoří T2_before (jen resampling do geometrie T1 bez registrace)
- sjednotí orientaci do RAS (aby se zobrazení shodovalo napříč SW)
- automaticky vybere "nejinformativnější" axiální řez podle ROI labelů v maskách (kontrola_masky.nii.gz):
  z* = řez s maximální plochou ROI; zobrazí z-2, z, z+2 (ošetřené okraje)
- spočítá metriky:
  * MI (Mattes) – přepočteno na kladné "vyšší = lepší" jako MI_pos = -MetricEvaluate
  * NCC (Pearson korelace v masce hlavy/mozku)
- vytvoří PDF:
  * souhrnné stránky (boxploty + scatter)
  * pro každého pacienta 2 stránky: overlay BEFORE/AFTER (2×3) a checkerboard BEFORE/AFTER (2×3)
- uloží CSV s metrikami

Požadavky:
pip install SimpleITK numpy matplotlib

Poznámka:
- očekává, že už máš vytvořené:
  derivatives/coregistered/sub-XXX/sub-XXX_T1w_N4.nii.gz
  derivatives/coregistered/sub-XXX/sub-XXX_T2w_reg.nii.gz
  derivatives/coregistered/sub-XXX/kontrola_masky.nii.gz
"""

In [3]:
import os
import csv
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from typing import List, Tuple, Dict, Set, Optional

# -----------------------
# UPRAV SI CESTY
# -----------------------
VYBRANE_TXT = r"vybrane_snimky.txt"                 # QC1 výstup: sub-XXX,T1_path,T2_path
PATIENT_FILTER_TXT = r"finalni_schvalene_masky.txt" # QC2 výstup: jen sub-XXX (volitelné)

COREG_DIR = r"D:\bakalarka\data_hie\derivatives\coregistered"

OUT_DIR = os.path.join(COREG_DIR, "_QC_registration_axial_roi")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_CSV = os.path.join(OUT_DIR, "registration_metrics.csv")
OUT_PDF = os.path.join(OUT_DIR, "registration_report.pdf")

CACHE_T2_N4 = True  # urychlí běh (uloží T2 po N4 pro QC do OUT_DIR)

# -----------------------
# ROI labely (tvé struktury zájmu z aseg)
# -----------------------
oblasti_zajmu = {
    10: "Levy_Thalamus", 49: "Pravy_Thalamus",
    17: "Levy_Hipokampus", 53: "Pravy_Hipokampus",
    11: "Levy_Caudate", 50: "Pravy_Caudate",
    12: "Levy_Putamen", 51: "Pravy_Putamen",
    13: "Levy_Pallidum", 52: "Pravy_Pallidum",
}
ROI_LABELS = sorted(list(oblasti_zajmu.keys()))

# -----------------------
# Helper funkce
# -----------------------
def odstran_stiny_n4(image: sitk.Image) -> sitk.Image:
    image_float = sitk.Cast(image, sitk.sitkFloat32)
    maska = sitk.OtsuThreshold(image_float, 0, 1, 200)
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    return corrector.Execute(image_float, maska)

def to_ras(img: sitk.Image) -> sitk.Image:
    return sitk.DICOMOrient(img, "RAS")

def robust_scale(arr2d: np.ndarray, p1: float = 1, p99: float = 99) -> np.ndarray:
    lo, hi = np.percentile(arr2d, [p1, p99])
    if hi <= lo:
        return np.zeros_like(arr2d, dtype=np.float32)
    x = np.clip(arr2d, lo, hi)
    return ((x - lo) / (hi - lo)).astype(np.float32)

def alpha_overlay(base01: np.ndarray, over01: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    return (1 - alpha) * base01 + alpha * over01

def load_pairs(vybrane_txt_path: str) -> List[Tuple[str, str, str]]:
    pairs: List[Tuple[str, str, str]] = []
    with open(vybrane_txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if "," not in line:
                raise ValueError(
                    f"{vybrane_txt_path} nevypadá jako 'sub-XXX,T1_path,T2_path'. "
                    f"Použij ho jako VYBRANE_TXT a seznam pacientů dej do PATIENT_FILTER_TXT."
                )
            pacient, t1_path, t2_path = line.split(",", 2)
            pairs.append((pacient, t1_path, t2_path))
    return pairs

def load_patient_filter(path: str) -> Set[str]:
    s: Set[str] = set()
    if path and os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    s.add(line)
    return s

def get_or_make_t2_clean_cache(pacient: str, t2_path: str) -> sitk.Image:
    cache_path = os.path.join(OUT_DIR, f"{pacient}_T2w_N4_forQC.nii.gz")
    if CACHE_T2_N4 and os.path.exists(cache_path):
        return sitk.ReadImage(cache_path)
    t2 = sitk.ReadImage(t2_path)
    t2_clean = odstran_stiny_n4(t2)
    if CACHE_T2_N4:
        sitk.WriteImage(t2_clean, cache_path)
    return t2_clean

def metric_MIpos_and_NCC(fixed: sitk.Image, moving: sitk.Image, mask: Optional[sitk.Image] = None) -> Tuple[float, float]:
    """
    MI: MetricEvaluate u ITK bývá ve smyslu minimalizace (často záporné hodnoty).
    Vracíme MI_pos = -MI_raw => vyšší = lepší.
    """
    fixed_f = sitk.Cast(fixed, sitk.sitkFloat32)
    moving_f = sitk.Cast(moving, sitk.sitkFloat32)

    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    if mask is not None:
        reg.SetMetricFixedMask(mask)

    mi_raw = reg.MetricEvaluate(fixed_f, moving_f)
    mi_pos = float(-mi_raw)

    f = sitk.GetArrayFromImage(fixed_f).astype(np.float32)
    m = sitk.GetArrayFromImage(moving_f).astype(np.float32)
    mk = sitk.GetArrayFromImage(mask).astype(bool) if mask is not None else np.ones_like(f, dtype=bool)

    f = f[mk]
    m = m[mk]
    if f.size < 10 or m.size < 10:
        return mi_pos, float("nan")

    f = f - f.mean()
    m = m - m.mean()
    denom = (np.sqrt((f * f).mean()) * np.sqrt((m * m).mean()))
    ncc = float((f * m).mean() / denom) if denom > 0 else float("nan")

    return mi_pos, ncc

def make_checkerboard(fixed_ras: sitk.Image, moving_ras: sitk.Image, squares=(8, 8, 8)) -> sitk.Image:
    return sitk.CheckerBoard(fixed_ras, moving_ras, squares)

def pick_best_axial_z_for_labels(mask_ras: sitk.Image, roi_labels: List[int]) -> Optional[int]:
    m = sitk.GetArrayFromImage(mask_ras).astype(np.int32)  # (z,y,x)
    roi = np.isin(m, np.array(roi_labels, dtype=np.int32))
    if roi.sum() == 0:
        return None
    return int(np.argmax(roi.sum(axis=(1, 2))))

def pick_three_z_around(z0: int, z_max: int, delta: int = 2) -> List[int]:
    cand = [z0 - delta, z0, z0 + delta]
    cand = [max(0, min(z_max, z)) for z in cand]
    cand = sorted(set(cand))

    while len(cand) < 3:
        if cand[0] > 0:
            cand = sorted(set([cand[0] - 1] + cand))
        elif cand[-1] < z_max:
            cand = sorted(set(cand + [cand[-1] + 1]))
        else:
            break

    return cand[:3]

def extract_axial_slices(img_ras: sitk.Image, z_list: List[int]) -> List[np.ndarray]:
    a = sitk.GetArrayFromImage(img_ras).astype(np.float32)  # (z,y,x)
    return [a[z, :, :] for z in z_list]

# -----------------------
# Hlavní běh
# -----------------------
def main():
    print("=== QC REGISTRATION REPORT (T2→T1) ===")
    print("VYBRANE_TXT:", VYBRANE_TXT)
    print("PATIENT_FILTER_TXT:", PATIENT_FILTER_TXT if os.path.exists(PATIENT_FILTER_TXT) else "(nenalezen – bez filtru)")
    print("COREG_DIR:", COREG_DIR)
    print("OUT_DIR:", OUT_DIR)
    print("OUT_PDF:", OUT_PDF)
    print("OUT_CSV:", OUT_CSV)
    print("CACHE_T2_N4:", CACHE_T2_N4)
    print("--------------------------------------")

    if not os.path.exists(VYBRANE_TXT):
        raise FileNotFoundError(f"Chybí {VYBRANE_TXT}")

    pairs_all = load_pairs(VYBRANE_TXT)
    print("Záznamů ve VYBRANE_TXT:", len(pairs_all))

    flt = load_patient_filter(PATIENT_FILTER_TXT)
    if flt:
        print("Pacientů ve filtru (QC2):", len(flt))
        pairs = [p for p in pairs_all if p[0] in flt]
    else:
        pairs = pairs_all

    print("Pacientů po filtraci:", len(pairs))
    print("--------------------------------------")

    rows: List[Dict[str, float]] = []
    processed: List[Tuple[str, str, str, str, str]] = []

    # PASS 1: metriky před/po
    print("PASS 1/2: Počítám metriky (MI_pos, NCC) pro BEFORE vs AFTER ...")
    skipped = 0
    for i, (pacient, _, t2_path) in enumerate(pairs, start=1):
        t1_n4_path = os.path.join(COREG_DIR, pacient, f"{pacient}_T1w_N4.nii.gz")
        t2_reg_path = os.path.join(COREG_DIR, pacient, f"{pacient}_T2w_reg.nii.gz")
        mask_path = os.path.join(COREG_DIR, pacient, "kontrola_masky.nii.gz")

        if not (os.path.exists(t1_n4_path) and os.path.exists(t2_reg_path) and os.path.exists(t2_path)):
            print(f"[SKIP {i}/{len(pairs)}] {pacient} – chybí T1_N4 nebo T2_reg nebo originální T2")
            skipped += 1
            continue

        print(f"[METRIKY {i}/{len(pairs)}] {pacient}")

        t1 = sitk.Cast(sitk.ReadImage(t1_n4_path), sitk.sitkFloat32)
        t2_clean = sitk.Cast(get_or_make_t2_clean_cache(pacient, t2_path), sitk.sitkFloat32)
        t2_after = sitk.Cast(sitk.ReadImage(t2_reg_path), sitk.sitkFloat32)

        t2_before = sitk.Resample(t2_clean, t1, sitk.Transform(), sitk.sitkLinear, 0.0, sitk.sitkFloat32)

        t1_ras = to_ras(t1)
        t2b_ras = to_ras(t2_before)
        t2a_ras = to_ras(t2_after)

        brain_mask = sitk.OtsuThreshold(t1_ras, 0, 1, 200)

        mi_b, ncc_b = metric_MIpos_and_NCC(t1_ras, t2b_ras, brain_mask)
        mi_a, ncc_a = metric_MIpos_and_NCC(t1_ras, t2a_ras, brain_mask)

        rows.append({
            "patient": pacient,
            "MI_before": mi_b,
            "MI_after": mi_a,
            "dMI": mi_a - mi_b,
            "NCC_before": ncc_b,
            "NCC_after": ncc_a,
            "dNCC": ncc_a - ncc_b
        })
        processed.append((pacient, t1_n4_path, t2_path, t2_reg_path, mask_path))

        print(f"         MI_pos: {mi_b:.3f} → {mi_a:.3f} (Δ {mi_a - mi_b:.3f}) | NCC: {ncc_b:.3f} → {ncc_a:.3f} (Δ {ncc_a - ncc_b:.3f})")

    if len(rows) == 0:
        raise RuntimeError("Nebyl zpracován žádný pacient (zkontroluj cesty a existenci souborů).")

    print("--------------------------------------")
    print(f"Hotovo PASS 1: zpracováno {len(rows)} pacientů, přeskočeno {skipped}.")
    print("Ukládám CSV:", OUT_CSV)

    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["patient", "MI_before", "MI_after", "dMI", "NCC_before", "NCC_after", "dNCC"]
        )
        writer.writeheader()
        for r in rows:
            writer.writerow(r)

    # Souhrnné array
    MI_b = np.array([r["MI_before"] for r in rows], dtype=float)
    MI_a = np.array([r["MI_after"] for r in rows], dtype=float)
    dMI = np.array([r["dMI"] for r in rows], dtype=float)

    NCC_b = np.array([r["NCC_before"] for r in rows], dtype=float)
    NCC_a = np.array([r["NCC_after"] for r in rows], dtype=float)
    dNCC = np.array([r["dNCC"] for r in rows], dtype=float)

    metrics_by_patient = {r["patient"]: r for r in rows}

    # PASS 2: PDF report
    print("--------------------------------------")
    print("PASS 2/2: Generuji PDF report ...")
    with PdfPages(OUT_PDF) as pdf:
        # Souhrn MI
        print("[PDF] Souhrnné grafy – MI_pos boxplot")
        fig = plt.figure(figsize=(11.7, 8.3))
        plt.title(f"Registrace T2→T1: Souhrn metrik (n={len(rows)}) | MI_pos vyšší = lepší")
        plt.boxplot([MI_b, MI_a], labels=["MI before", "MI after"])
        plt.ylabel("MI_pos (= -MI_raw z ITK)")
        plt.grid(True, axis="y", linestyle="--", alpha=0.4)
        plt.figtext(
            0.02, 0.02,
            f"MI_pos: median(before)={np.median(MI_b):.3f}, median(after)={np.median(MI_a):.3f}, "
            f"improvement%={(np.mean(dMI > 0) * 100):.1f}%",
            ha="left", va="bottom"
        )
        pdf.savefig(fig); plt.close(fig)

        # Souhrn NCC
        print("[PDF] Souhrnné grafy – NCC boxplot")
        fig = plt.figure(figsize=(11.7, 8.3))
        plt.title(f"Registrace T2→T1: Souhrn metrik (n={len(rows)})")
        plt.boxplot([NCC_b, NCC_a], labels=["NCC before", "NCC after"])
        plt.ylabel("NCC (vyšší = lepší)")
        plt.grid(True, axis="y", linestyle="--", alpha=0.4)
        plt.figtext(
            0.02, 0.02,
            f"NCC: median(before)={np.nanmedian(NCC_b):.3f}, median(after)={np.nanmedian(NCC_a):.3f}, "
            f"improvement%={(np.mean(dNCC > 0) * 100):.1f}%",
            ha="left", va="bottom"
        )
        pdf.savefig(fig); plt.close(fig)

        # Scatter MI
        print("[PDF] Souhrnné grafy – MI_pos scatter before vs after")
        fig = plt.figure(figsize=(11.7, 8.3))
        plt.title("MI_pos: before vs after (každý bod = pacient)")
        plt.scatter(MI_b, MI_a)
        mn = min(MI_b.min(), MI_a.min())
        mx = max(MI_b.max(), MI_a.max())
        plt.plot([mn, mx], [mn, mx], linestyle="--")
        plt.xlabel("MI before")
        plt.ylabel("MI after")
        plt.grid(True, linestyle="--", alpha=0.3)
        pdf.savefig(fig); plt.close(fig)

        # Scatter NCC
        print("[PDF] Souhrnné grafy – NCC scatter before vs after")
        fig = plt.figure(figsize=(11.7, 8.3))
        plt.title("NCC: before vs after (každý bod = pacient)")
        plt.scatter(NCC_b, NCC_a)
        mn = np.nanmin([np.nanmin(NCC_b), np.nanmin(NCC_a)])
        mx = np.nanmax([np.nanmax(NCC_b), np.nanmax(NCC_a)])
        plt.plot([mn, mx], [mn, mx], linestyle="--")
        plt.xlabel("NCC before")
        plt.ylabel("NCC after")
        plt.grid(True, linestyle="--", alpha=0.3)
        pdf.savefig(fig); plt.close(fig)

        # Per-patient stránky (axiál 3 řezy)
        for i, (pacient, t1_n4_path, t2_path, t2_reg_path, mask_path) in enumerate(processed, start=1):
            m = metrics_by_patient[pacient]
            print(f"[PDF {i}/{len(processed)}] {pacient} – načítám data, vybírám z* podle ROI, renderuji stránky")

            t1 = sitk.Cast(sitk.ReadImage(t1_n4_path), sitk.sitkFloat32)
            t2_clean = sitk.Cast(get_or_make_t2_clean_cache(pacient, t2_path), sitk.sitkFloat32)
            t2_after = sitk.Cast(sitk.ReadImage(t2_reg_path), sitk.sitkFloat32)
            t2_before = sitk.Resample(t2_clean, t1, sitk.Transform(), sitk.sitkLinear, 0.0, sitk.sitkFloat32)

            t1_ras = to_ras(t1)
            t2b_ras = to_ras(t2_before)
            t2a_ras = to_ras(t2_after)

            z_dim = sitk.GetArrayFromImage(t1_ras).shape[0]
            z0 = z_dim // 2

            if os.path.exists(mask_path):
                mask_img = sitk.ReadImage(mask_path)
                mask_ras = to_ras(mask_img)
                z_best = pick_best_axial_z_for_labels(mask_ras, ROI_LABELS)
                if z_best is not None:
                    z0 = z_best
            else:
                print(f"         [WARN] {pacient} – chybí kontrola_masky.nii.gz, beru středový řez")

            z_list = pick_three_z_around(z0, z_max=z_dim - 1, delta=2)
            print(f"         z*={z0}, z_list={z_list}")

            # OVERLAY (2×3)
            t1_slices = extract_axial_slices(t1_ras, z_list)
            t2b_slices = extract_axial_slices(t2b_ras, z_list)
            t2a_slices = extract_axial_slices(t2a_ras, z_list)

            t1_s = [robust_scale(s) for s in t1_slices]
            t2b_s = [robust_scale(s) for s in t2b_slices]
            t2a_s = [robust_scale(s) for s in t2a_slices]

            ov_b = [alpha_overlay(t1_s[j], t2b_s[j], alpha=0.45) for j in range(len(z_list))]
            ov_a = [alpha_overlay(t1_s[j], t2a_s[j], alpha=0.45) for j in range(len(z_list))]

            fig, axes = plt.subplots(2, 3, figsize=(11.7, 8.3))
            for j, z in enumerate(z_list):
                axes[0, j].imshow(ov_b[j], cmap="gray")
                axes[0, j].set_title(f"Overlay BEFORE | Axial z={z}")
                axes[0, j].axis("off")

                axes[1, j].imshow(ov_a[j], cmap="gray")
                axes[1, j].set_title(f"Overlay AFTER | Axial z={z}")
                axes[1, j].axis("off")

            fig.suptitle(
                f"{pacient} | ROI: BG+Thalamus+Hippocampus | z*={z0} (z±2) | "
                f"MI: {m['MI_before']:.3f} → {m['MI_after']:.3f} (Δ {m['dMI']:.3f}) | "
                f"NCC: {m['NCC_before']:.3f} → {m['NCC_after']:.3f} (Δ {m['dNCC']:.3f})",
                fontsize=12
            )
            plt.tight_layout()
            pdf.savefig(fig); plt.close(fig)

            # CHECKERBOARD (2×3)
            cb_b = to_ras(sitk.Cast(make_checkerboard(t1_ras, t2b_ras, squares=(8, 8, 8)), sitk.sitkFloat32))
            cb_a = to_ras(sitk.Cast(make_checkerboard(t1_ras, t2a_ras, squares=(8, 8, 8)), sitk.sitkFloat32))

            cb_b_slices = extract_axial_slices(cb_b, z_list)
            cb_a_slices = extract_axial_slices(cb_a, z_list)

            cb_b_s = [robust_scale(s) for s in cb_b_slices]
            cb_a_s = [robust_scale(s) for s in cb_a_slices]

            fig, axes = plt.subplots(2, 3, figsize=(11.7, 8.3))
            for j, z in enumerate(z_list):
                axes[0, j].imshow(cb_b_s[j], cmap="gray")
                axes[0, j].set_title(f"Checkerboard BEFORE | Axial z={z}")
                axes[0, j].axis("off")

                axes[1, j].imshow(cb_a_s[j], cmap="gray")
                axes[1, j].set_title(f"Checkerboard AFTER | Axial z={z}")
                axes[1, j].axis("off")

            fig.suptitle(f"{pacient} | Checkerboard kontrola | řez zvolen podle ROI (z*={z0})", fontsize=12)
            plt.tight_layout()
            pdf.savefig(fig); plt.close(fig)

    print("--------------------------------------")
    print("HOTOVO")
    print("PDF:", OUT_PDF)
    print("CSV:", OUT_CSV)
    print("OUT_DIR:", OUT_DIR)


if __name__ == "__main__":
    main()

=== QC REGISTRATION REPORT (T2→T1) ===
VYBRANE_TXT: vybrane_snimky.txt
PATIENT_FILTER_TXT: finalni_schvalene_masky.txt
COREG_DIR: D:\bakalarka\data_hie\derivatives\coregistered
OUT_DIR: D:\bakalarka\data_hie\derivatives\coregistered\_QC_registration_axial_roi
OUT_PDF: D:\bakalarka\data_hie\derivatives\coregistered\_QC_registration_axial_roi\registration_report.pdf
OUT_CSV: D:\bakalarka\data_hie\derivatives\coregistered\_QC_registration_axial_roi\registration_metrics.csv
CACHE_T2_N4: True
--------------------------------------
Záznamů ve VYBRANE_TXT: 45
Pacientů ve filtru (QC2): 41
Pacientů po filtraci: 41
--------------------------------------
PASS 1/2: Počítám metriky (MI_pos, NCC) pro BEFORE vs AFTER ...
[METRIKY 1/41] sub-002
         MI_pos: 0.077 → 0.118 (Δ 0.041) | NCC: 0.033 → 0.121 (Δ 0.089)
[METRIKY 2/41] sub-004
         MI_pos: 0.126 → 0.111 (Δ -0.015) | NCC: 0.275 → 0.265 (Δ -0.009)
[METRIKY 3/41] sub-005
         MI_pos: 0.071 → 0.054 (Δ -0.017) | NCC: -0.012 → -0.008 (Δ 0

C:\Users\42073\AppData\Local\Temp\ipykernel_17372\3234338286.py:267: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot([MI_b, MI_a], labels=["MI before", "MI after"])


[PDF] Souhrnné grafy – NCC boxplot
[PDF] Souhrnné grafy – MI_pos scatter before vs after


C:\Users\42073\AppData\Local\Temp\ipykernel_17372\3234338286.py:282: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot([NCC_b, NCC_a], labels=["NCC before", "NCC after"])


[PDF] Souhrnné grafy – NCC scatter before vs after
[PDF 1/41] sub-002 – načítám data, vybírám z* podle ROI, renderuji stránky
         z*=113, z_list=[111, 113, 115]
[PDF 2/41] sub-004 – načítám data, vybírám z* podle ROI, renderuji stránky
         z*=112, z_list=[110, 112, 114]
[PDF 3/41] sub-005 – načítám data, vybírám z* podle ROI, renderuji stránky
         z*=121, z_list=[119, 121, 123]
[PDF 4/41] sub-006 – načítám data, vybírám z* podle ROI, renderuji stránky
         z*=114, z_list=[112, 114, 116]
[PDF 5/41] sub-007 – načítám data, vybírám z* podle ROI, renderuji stránky
         z*=121, z_list=[119, 121, 123]
[PDF 6/41] sub-009 – načítám data, vybírám z* podle ROI, renderuji stránky
         z*=110, z_list=[108, 110, 112]
[PDF 7/41] sub-011 – načítám data, vybírám z* podle ROI, renderuji stránky
         z*=120, z_list=[118, 120, 122]
[PDF 8/41] sub-012 – načítám data, vybírám z* podle ROI, renderuji stránky
         z*=117, z_list=[115, 117, 119]
[PDF 9/41] sub-013 – načítám 

"""
Regenerace PDF QC reportu (s opravou "palačinek" přes voxel spacing)
- NEPŘEPOČÍTÁVÁ N4 T2 (použije existující cache sub-XXX_T2w_N4_forQC.nii.gz)
- volitelně znovu NEPOČÍTÁ ani metriky (načte je z existujícího CSV), a jen překreslí PDF

Co opravuje "palačinky":
- matplotlib imshow ignoruje spacing; proto nastavujeme aspect = sy/sx pro axiální řezy.

Požadavky:
pip install SimpleITK numpy matplotlib
"""

In [4]:
import os
import csv
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from typing import List, Tuple, Dict, Set, Optional


# =======================
# UPRAV SI CESTY
# =======================
VYBRANE_TXT = r"vybrane_snimky.txt"                 # QC1: sub-XXX,T1_path,T2_path
PATIENT_FILTER_TXT = r"finalni_schvalene_masky.txt" # QC2: sub-XXX (volitelné)

COREG_DIR = r"D:\bakalarka\data_hie\derivatives\coregistered"

# musí být stejné OUT_DIR jako v původním běhu (kvůli cache T2 N4 a CSV)
OUT_DIR = os.path.join(COREG_DIR, "_QC_registration_axial_roi")
os.makedirs(OUT_DIR, exist_ok=True)

# nový PDF (aby sis nepřepsal původní – klidně změň na původní název, pokud chceš overwrite)
OUT_PDF = os.path.join(OUT_DIR, "registration_report_ASPECT_FIXED.pdf")

# existující metriky z předchozího běhu (pokud existují, načtou se a metriky se nepřepočítají)
METRICS_CSV = os.path.join(OUT_DIR, "registration_metrics.csv")

# chování
REUSE_METRICS_CSV = True          # True = pokud existuje CSV, nepřepočítá metriky
REQUIRE_T2_N4_CACHE = True        # True = pokud chybí cache T2 N4, pacienta přeskočí (nepočítá N4)
CACHE_FILENAME_FMT = "{patient}_T2w_N4_forQC.nii.gz"


# =======================
# ROI labely
# =======================
oblasti_zajmu = {
    10: "Levy_Thalamus", 49: "Pravy_Thalamus",
    17: "Levy_Hipokampus", 53: "Pravy_Hipokampus",
    11: "Levy_Caudate", 50: "Pravy_Caudate",
    12: "Levy_Putamen", 51: "Pravy_Putamen",
    13: "Levy_Pallidum", 52: "Pravy_Pallidum",
}
ROI_LABELS = sorted(list(oblasti_zajmu.keys()))


# =======================
# Helper funkce
# =======================
def to_ras(img: sitk.Image) -> sitk.Image:
    return sitk.DICOMOrient(img, "RAS")

def robust_scale(arr2d: np.ndarray, p1: float = 1, p99: float = 99) -> np.ndarray:
    lo, hi = np.percentile(arr2d, [p1, p99])
    if hi <= lo:
        return np.zeros_like(arr2d, dtype=np.float32)
    x = np.clip(arr2d, lo, hi)
    return ((x - lo) / (hi - lo)).astype(np.float32)

def alpha_overlay(base01: np.ndarray, over01: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    return (1 - alpha) * base01 + alpha * over01

def load_pairs(vybrane_txt_path: str) -> List[Tuple[str, str, str]]:
    pairs: List[Tuple[str, str, str]] = []
    with open(vybrane_txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if "," not in line:
                raise ValueError(
                    f"{vybrane_txt_path} nevypadá jako 'sub-XXX,T1_path,T2_path'. "
                    f"Použij vybrane_snimky.txt jako VYBRANE_TXT."
                )
            pacient, t1_path, t2_path = line.split(",", 2)
            pairs.append((pacient, t1_path, t2_path))
    return pairs

def load_patient_filter(path: str) -> Set[str]:
    s: Set[str] = set()
    if path and os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    s.add(line)
    return s

def load_metrics_csv(path: str) -> Dict[str, Dict[str, float]]:
    """
    očekává sloupce: patient, MI_before, MI_after, dMI, NCC_before, NCC_after, dNCC
    """
    out: Dict[str, Dict[str, float]] = {}
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            p = row["patient"]
            out[p] = {
                "MI_before": float(row["MI_before"]),
                "MI_after": float(row["MI_after"]),
                "dMI": float(row["dMI"]),
                "NCC_before": float(row["NCC_before"]),
                "NCC_after": float(row["NCC_after"]),
                "dNCC": float(row["dNCC"]),
            }
    return out

def metric_MIpos_and_NCC(fixed: sitk.Image, moving: sitk.Image, mask: Optional[sitk.Image] = None):
    """
    MI_pos = - MetricEvaluate (vyšší = lepší)
    NCC = Pearson korelace v masce
    """
    fixed_f = sitk.Cast(fixed, sitk.sitkFloat32)
    moving_f = sitk.Cast(moving, sitk.sitkFloat32)

    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    if mask is not None:
        reg.SetMetricFixedMask(mask)
    mi_raw = reg.MetricEvaluate(fixed_f, moving_f)
    mi_pos = float(-mi_raw)

    f = sitk.GetArrayFromImage(fixed_f).astype(np.float32)
    m = sitk.GetArrayFromImage(moving_f).astype(np.float32)
    mk = sitk.GetArrayFromImage(mask).astype(bool) if mask is not None else np.ones_like(f, dtype=bool)

    f = f[mk]; m = m[mk]
    if f.size < 10 or m.size < 10:
        return mi_pos, float("nan")

    f = f - f.mean()
    m = m - m.mean()
    denom = (np.sqrt((f * f).mean()) * np.sqrt((m * m).mean()))
    ncc = float((f * m).mean() / denom) if denom > 0 else float("nan")
    return mi_pos, ncc

def make_checkerboard(fixed_ras: sitk.Image, moving_ras: sitk.Image, squares=(8, 8, 8)) -> sitk.Image:
    return sitk.CheckerBoard(fixed_ras, moving_ras, squares)

def pick_best_axial_z_for_labels(mask_ras: sitk.Image, roi_labels: List[int]) -> Optional[int]:
    m = sitk.GetArrayFromImage(mask_ras).astype(np.int32)  # (z,y,x)
    roi = np.isin(m, np.array(roi_labels, dtype=np.int32))
    if roi.sum() == 0:
        return None
    return int(np.argmax(roi.sum(axis=(1, 2))))

def pick_three_z_around(z0: int, z_max: int, delta: int = 2) -> List[int]:
    cand = [z0 - delta, z0, z0 + delta]
    cand = [max(0, min(z_max, z)) for z in cand]
    cand = sorted(set(cand))
    while len(cand) < 3:
        if cand[0] > 0:
            cand = sorted(set([cand[0] - 1] + cand))
        elif cand[-1] < z_max:
            cand = sorted(set(cand + [cand[-1] + 1]))
        else:
            break
    return cand[:3]

def extract_axial_slices(img_ras: sitk.Image, z_list: List[int]) -> List[np.ndarray]:
    a = sitk.GetArrayFromImage(img_ras).astype(np.float32)  # (z,y,x)
    return [a[z, :, :] for z in z_list]

def load_t2_clean_from_cache(patient: str) -> Optional[sitk.Image]:
    cache_path = os.path.join(OUT_DIR, CACHE_FILENAME_FMT.format(patient=patient))
    if os.path.exists(cache_path):
        return sitk.ReadImage(cache_path)
    return None


# =======================
# Hlavní běh
# =======================
def main():
    print("=== REGENERACE PDF (ASPECT FIX) ===")
    print("OUT_DIR:", OUT_DIR)
    print("OUT_PDF:", OUT_PDF)
    print("METRICS_CSV:", METRICS_CSV, "| reuse:", REUSE_METRICS_CSV)
    print("REQUIRE_T2_N4_CACHE:", REQUIRE_T2_N4_CACHE)
    print("----------------------------------")

    pairs_all = load_pairs(VYBRANE_TXT)
    flt = load_patient_filter(PATIENT_FILTER_TXT)
    pairs = [p for p in pairs_all if p[0] in flt] if flt else pairs_all
    print("Pacientů po filtraci:", len(pairs))

    # připrav seznam pacientů s existujícími soubory
    processed: List[Tuple[str, str, str, str, str]] = []
    for pacient, _, t2_path in pairs:
        t1_n4_path = os.path.join(COREG_DIR, pacient, f"{pacient}_T1w_N4.nii.gz")
        t2_reg_path = os.path.join(COREG_DIR, pacient, f"{pacient}_T2w_reg.nii.gz")
        mask_path = os.path.join(COREG_DIR, pacient, "kontrola_masky.nii.gz")

        if not (os.path.exists(t1_n4_path) and os.path.exists(t2_reg_path) and os.path.exists(t2_path)):
            print(f"[SKIP] {pacient} – chybí T1_N4 nebo T2_reg nebo originální T2")
            continue

        if REQUIRE_T2_N4_CACHE:
            t2_clean = load_t2_clean_from_cache(pacient)
            if t2_clean is None:
                print(f"[SKIP] {pacient} – chybí cache {CACHE_FILENAME_FMT.format(patient=pacient)} (nepočítám N4)")
                continue

        processed.append((pacient, t1_n4_path, t2_path, t2_reg_path, mask_path))

    print("Zpracováno pro PDF:", len(processed))
    if len(processed) == 0:
        raise RuntimeError("Není co renderovat (všechno přeskočeno).")

    # metriky: načti z CSV nebo přepočítej (bez N4; používá cache)
    metrics_by_patient: Dict[str, Dict[str, float]] = {}
    if REUSE_METRICS_CSV and os.path.exists(METRICS_CSV):
        print("Načítám metriky z CSV...")
        metrics_by_patient = load_metrics_csv(METRICS_CSV)
    else:
        print("CSV metrik nenalezeno / reuse vypnuto -> přepočítávám metriky...")
        for i, (pacient, t1_n4_path, t2_path, t2_reg_path, _) in enumerate(processed, start=1):
            print(f"[METRIKY {i}/{len(processed)}] {pacient}")
            t1 = sitk.Cast(sitk.ReadImage(t1_n4_path), sitk.sitkFloat32)

            t2_clean = load_t2_clean_from_cache(pacient)
            if t2_clean is None:
                raise RuntimeError(f"Chybí cache pro {pacient} (a REQUIRE_T2_N4_CACHE je False/True dle nastavení).")
            t2_clean = sitk.Cast(t2_clean, sitk.sitkFloat32)

            t2_after = sitk.Cast(sitk.ReadImage(t2_reg_path), sitk.sitkFloat32)
            t2_before = sitk.Resample(t2_clean, t1, sitk.Transform(), sitk.sitkLinear, 0.0, sitk.sitkFloat32)

            t1_ras = to_ras(t1)
            t2b_ras = to_ras(t2_before)
            t2a_ras = to_ras(t2_after)

            brain_mask = sitk.OtsuThreshold(t1_ras, 0, 1, 200)
            mi_b, ncc_b = metric_MIpos_and_NCC(t1_ras, t2b_ras, brain_mask)
            mi_a, ncc_a = metric_MIpos_and_NCC(t1_ras, t2a_ras, brain_mask)

            metrics_by_patient[pacient] = {
                "MI_before": mi_b, "MI_after": mi_a, "dMI": mi_a - mi_b,
                "NCC_before": ncc_b, "NCC_after": ncc_a, "dNCC": ncc_a - ncc_b,
            }

    # připrav souhrnné vektory (jen pacienti, co mají metriky)
    rows = []
    for pacient, *_ in processed:
        if pacient in metrics_by_patient:
            r = metrics_by_patient[pacient].copy()
            r["patient"] = pacient
            rows.append(r)

    MI_b = np.array([r["MI_before"] for r in rows], dtype=float)
    MI_a = np.array([r["MI_after"] for r in rows], dtype=float)
    dMI = np.array([r["dMI"] for r in rows], dtype=float)

    NCC_b = np.array([r["NCC_before"] for r in rows], dtype=float)
    NCC_a = np.array([r["NCC_after"] for r in rows], dtype=float)
    dNCC = np.array([r["dNCC"] for r in rows], dtype=float)

    # render PDF
    print("----------------------------------")
    print("Generuji PDF (s aspect fixem)...")
    with PdfPages(OUT_PDF) as pdf:
        # Souhrn MI
        fig = plt.figure(figsize=(11.7, 8.3))
        plt.title(f"Registrace T2→T1: Souhrn metrik (n={len(rows)}) | MI_pos vyšší = lepší")
        plt.boxplot([MI_b, MI_a], labels=["MI before", "MI after"])
        plt.ylabel("MI_pos (= -MI_raw z ITK)")
        plt.grid(True, axis="y", linestyle="--", alpha=0.4)
        plt.figtext(
            0.02, 0.02,
            f"MI_pos: median(before)={np.median(MI_b):.3f}, median(after)={np.median(MI_a):.3f}, "
            f"improvement%={(np.mean(dMI > 0) * 100):.1f}%",
            ha="left", va="bottom"
        )
        pdf.savefig(fig); plt.close(fig)

        # Souhrn NCC
        fig = plt.figure(figsize=(11.7, 8.3))
        plt.title(f"Registrace T2→T1: Souhrn metrik (n={len(rows)})")
        plt.boxplot([NCC_b, NCC_a], labels=["NCC before", "NCC after"])
        plt.ylabel("NCC (vyšší = lepší)")
        plt.grid(True, axis="y", linestyle="--", alpha=0.4)
        plt.figtext(
            0.02, 0.02,
            f"NCC: median(before)={np.nanmedian(NCC_b):.3f}, median(after)={np.nanmedian(NCC_a):.3f}, "
            f"improvement%={(np.mean(dNCC > 0) * 100):.1f}%",
            ha="left", va="bottom"
        )
        pdf.savefig(fig); plt.close(fig)

        # Scatter MI
        fig = plt.figure(figsize=(11.7, 8.3))
        plt.title("MI_pos: before vs after (každý bod = pacient)")
        plt.scatter(MI_b, MI_a)
        mn = min(MI_b.min(), MI_a.min())
        mx = max(MI_b.max(), MI_a.max())
        plt.plot([mn, mx], [mn, mx], linestyle="--")
        plt.xlabel("MI before")
        plt.ylabel("MI after")
        plt.grid(True, linestyle="--", alpha=0.3)
        pdf.savefig(fig); plt.close(fig)

        # Scatter NCC
        fig = plt.figure(figsize=(11.7, 8.3))
        plt.title("NCC: before vs after (každý bod = pacient)")
        plt.scatter(NCC_b, NCC_a)
        mn = np.nanmin([np.nanmin(NCC_b), np.nanmin(NCC_a)])
        mx = np.nanmax([np.nanmax(NCC_b), np.nanmax(NCC_a)])
        plt.plot([mn, mx], [mn, mx], linestyle="--")
        plt.xlabel("NCC before")
        plt.ylabel("NCC after")
        plt.grid(True, linestyle="--", alpha=0.3)
        pdf.savefig(fig); plt.close(fig)

        # Per-patient stránky
        for i, (pacient, t1_n4_path, t2_path, t2_reg_path, mask_path) in enumerate(processed, start=1):
            print(f"[PDF {i}/{len(processed)}] {pacient}")

            t1 = sitk.Cast(sitk.ReadImage(t1_n4_path), sitk.sitkFloat32)
            t2_clean = load_t2_clean_from_cache(pacient)
            if t2_clean is None:
                if REQUIRE_T2_N4_CACHE:
                    print(f"   [SKIP] chybí cache T2 N4")
                    continue
                else:
                    raise RuntimeError(f"Chybí cache pro {pacient}.")
            t2_clean = sitk.Cast(t2_clean, sitk.sitkFloat32)

            t2_after = sitk.Cast(sitk.ReadImage(t2_reg_path), sitk.sitkFloat32)
            t2_before = sitk.Resample(t2_clean, t1, sitk.Transform(), sitk.sitkLinear, 0.0, sitk.sitkFloat32)

            t1_ras = to_ras(t1)
            t2b_ras = to_ras(t2_before)
            t2a_ras = to_ras(t2_after)

            # ====== ASPECT FIX (pro axiální řezy) ======
            sx, sy, sz = t1_ras.GetSpacing()       # (x,y,z)
            ax_aspect = sy / sx                    # pro 2D (y,x)
            # =========================================

            z_dim = sitk.GetArrayFromImage(t1_ras).shape[0]
            z0 = z_dim // 2
            if os.path.exists(mask_path):
                mask_ras = to_ras(sitk.ReadImage(mask_path))
                z_best = pick_best_axial_z_for_labels(mask_ras, ROI_LABELS)
                if z_best is not None:
                    z0 = z_best

            z_list = pick_three_z_around(z0, z_max=z_dim - 1, delta=2)

            # metriky do titulku (pokud chybí, dáme NaN)
            mm = metrics_by_patient.get(pacient, {
                "MI_before": float("nan"), "MI_after": float("nan"), "dMI": float("nan"),
                "NCC_before": float("nan"), "NCC_after": float("nan"), "dNCC": float("nan"),
            })

            # OVERLAY (2×3)
            t1_slices = extract_axial_slices(t1_ras, z_list)
            t2b_slices = extract_axial_slices(t2b_ras, z_list)
            t2a_slices = extract_axial_slices(t2a_ras, z_list)

            t1_s = [robust_scale(s) for s in t1_slices]
            t2b_s = [robust_scale(s) for s in t2b_slices]
            t2a_s = [robust_scale(s) for s in t2a_slices]

            ov_b = [alpha_overlay(t1_s[j], t2b_s[j], alpha=0.45) for j in range(len(z_list))]
            ov_a = [alpha_overlay(t1_s[j], t2a_s[j], alpha=0.45) for j in range(len(z_list))]

            fig, axes = plt.subplots(2, 3, figsize=(11.7, 8.3))
            for j, z in enumerate(z_list):
                axes[0, j].imshow(ov_b[j], cmap="gray", aspect=ax_aspect)
                axes[0, j].set_title(f"Overlay BEFORE | Axial z={z}")
                axes[0, j].axis("off")

                axes[1, j].imshow(ov_a[j], cmap="gray", aspect=ax_aspect)
                axes[1, j].set_title(f"Overlay AFTER | Axial z={z}")
                axes[1, j].axis("off")

            fig.suptitle(
                f"{pacient} | spacing=({sx:.3f},{sy:.3f},{sz:.3f}) | z*={z0} (z±2) | "
                f"MI: {mm['MI_before']:.3f} → {mm['MI_after']:.3f} (Δ {mm['dMI']:.3f}) | "
                f"NCC: {mm['NCC_before']:.3f} → {mm['NCC_after']:.3f} (Δ {mm['dNCC']:.3f})",
                fontsize=12
            )
            plt.tight_layout()
            pdf.savefig(fig); plt.close(fig)

            # CHECKERBOARD (2×3)
            cb_b = to_ras(sitk.Cast(make_checkerboard(t1_ras, t2b_ras, squares=(8, 8, 8)), sitk.sitkFloat32))
            cb_a = to_ras(sitk.Cast(make_checkerboard(t1_ras, t2a_ras, squares=(8, 8, 8)), sitk.sitkFloat32))

            cb_b_slices = extract_axial_slices(cb_b, z_list)
            cb_a_slices = extract_axial_slices(cb_a, z_list)

            cb_b_s = [robust_scale(s) for s in cb_b_slices]
            cb_a_s = [robust_scale(s) for s in cb_a_slices]

            fig, axes = plt.subplots(2, 3, figsize=(11.7, 8.3))
            for j, z in enumerate(z_list):
                axes[0, j].imshow(cb_b_s[j], cmap="gray", aspect=ax_aspect)
                axes[0, j].set_title(f"Checkerboard BEFORE | Axial z={z}")
                axes[0, j].axis("off")

                axes[1, j].imshow(cb_a_s[j], cmap="gray", aspect=ax_aspect)
                axes[1, j].set_title(f"Checkerboard AFTER | Axial z={z}")
                axes[1, j].axis("off")

            fig.suptitle(
                f"{pacient} | Checkerboard kontrola | spacing=({sx:.3f},{sy:.3f},{sz:.3f}) | z*={z0}",
                fontsize=12
            )
            plt.tight_layout()
            pdf.savefig(fig); plt.close(fig)

    print("----------------------------------")
    print("HOTOVO")
    print("Nové PDF:", OUT_PDF)


if __name__ == "__main__":
    main()

=== REGENERACE PDF (ASPECT FIX) ===
OUT_DIR: D:\bakalarka\data_hie\derivatives\coregistered\_QC_registration_axial_roi
OUT_PDF: D:\bakalarka\data_hie\derivatives\coregistered\_QC_registration_axial_roi\registration_report_ASPECT_FIXED.pdf
METRICS_CSV: D:\bakalarka\data_hie\derivatives\coregistered\_QC_registration_axial_roi\registration_metrics.csv | reuse: True
REQUIRE_T2_N4_CACHE: True
----------------------------------
Pacientů po filtraci: 41
Zpracováno pro PDF: 41
Načítám metriky z CSV...
----------------------------------
Generuji PDF (s aspect fixem)...


C:\Users\42073\AppData\Local\Temp\ipykernel_17372\818056659.py:265: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot([MI_b, MI_a], labels=["MI before", "MI after"])
C:\Users\42073\AppData\Local\Temp\ipykernel_17372\818056659.py:279: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot([NCC_b, NCC_a], labels=["NCC before", "NCC after"])


[PDF 1/41] sub-002
[PDF 2/41] sub-004
[PDF 3/41] sub-005
[PDF 4/41] sub-006
[PDF 5/41] sub-007
[PDF 6/41] sub-009
[PDF 7/41] sub-011
[PDF 8/41] sub-012
[PDF 9/41] sub-013
[PDF 10/41] sub-015
[PDF 11/41] sub-016
[PDF 12/41] sub-017
[PDF 13/41] sub-018
[PDF 14/41] sub-019
[PDF 15/41] sub-020
[PDF 16/41] sub-021
[PDF 17/41] sub-022
[PDF 18/41] sub-023
[PDF 19/41] sub-024
[PDF 20/41] sub-026
[PDF 21/41] sub-027
[PDF 22/41] sub-028
[PDF 23/41] sub-029
[PDF 24/41] sub-030
[PDF 25/41] sub-031
[PDF 26/41] sub-032
[PDF 27/41] sub-033
[PDF 28/41] sub-034
[PDF 29/41] sub-035
[PDF 30/41] sub-036
[PDF 31/41] sub-037
[PDF 32/41] sub-039
[PDF 33/41] sub-040
[PDF 34/41] sub-041
[PDF 35/41] sub-042
[PDF 36/41] sub-043
[PDF 37/41] sub-044
[PDF 38/41] sub-045
[PDF 39/41] sub-046
[PDF 40/41] sub-047
[PDF 41/41] sub-048
----------------------------------
HOTOVO
Nové PDF: D:\bakalarka\data_hie\derivatives\coregistered\_QC_registration_axial_roi\registration_report_ASPECT_FIXED.pdf
